# [Chapter 5 — SIR_I Derivation, §5.2-5.4] Assembling the SIR_I Model from Compartment Equations

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 5 — SIR_I Derivation
**Considerations developed:** 2 (compartmental simplifications), 4 (force of vs from)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Derives the SIR_I system from compartment-by-compartment bookkeeping. Introduces the infected-viewpoint formulation $dI/dt = \alpha I - I/\tau_R$ that the book uses throughout, and shows the derivation produces the same dynamics as the susceptible-viewpoint form.

## What you should already know
Chapter 2 series notebooks; basic understanding of ODEs.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_05
from shared.seeds import set_seed_chapter_05
from scipy.integrate import solve_ivp

set_seed_chapter_05()
book_style()


## The SIR_I system (infected viewpoint)

For each compartment, change = (rate in) − (rate out):

- **Susceptible:** $\frac{dS}{dt} = -\alpha I$  (loses individuals as they become infected; $\alpha = c_I \beta P_S$)
- **Infectious:** $\frac{dI}{dt} = \alpha I - \frac{I}{\tau_R}$  (gains from new infections, loses to recovery)
- **Recovered:** $\frac{dR}{dt} = \frac{I}{\tau_R}$  (gains from recoveries)

with the conservation $S + I + R = N$ holding for all $t$ (no demographic turnover).

The $\alpha I$ term is *the* infection-equation; it equals $\lambda S$ identically when $c_S = c_I$, and the two forms produce identical dynamics regardless. The book uses the $\alpha I$ form because $\alpha$ is what data observe (per-infectious incidence).

In [ ]:
# Verify equivalence of two formulations numerically
params = baseline_chapter_05()
N = params['N']
beta = params['beta']
tau_R = params['tau_R']
c_I = params['c_I']

# Both formulations need c_S; for verification, set c_S = c_I (otherwise they differ as expected)
c_S = c_I

# Initial conditions
y0 = [0.999, 0.001, 0.0]  # S, I, R
t_span = (0, 100)
t_eval = np.linspace(0, 100, 1001)

# Form A (alpha I formulation)
def rhs_alpha(t, y):
    S, I, R = y
    P_S = S / N
    alpha = c_I * beta * P_S
    return [-alpha * I, alpha * I - I / tau_R, I / tau_R]

# Form B (lambda S formulation)
def rhs_lambda(t, y):
    S, I, R = y
    P_I = I / N
    lambda_ = c_S * beta * P_I
    return [-lambda_ * S, lambda_ * S - I / tau_R, I / tau_R]

sol_A = solve_ivp(rhs_alpha, t_span, y0, t_eval=t_eval, rtol=1e-10, atol=1e-12)
sol_B = solve_ivp(rhs_lambda, t_span, y0, t_eval=t_eval, rtol=1e-10, atol=1e-12)

# Verify they produce identical trajectories
max_diff = max(np.abs(sol_A.y - sol_B.y).max(), 0)
print(f"Max difference between alpha and lambda formulations: {max_diff:.2e}")
print(f"(Should be effectively zero when c_S = c_I.)")


In [ ]:
# Plot the standard SIR_I trajectory
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(sol_A.t, sol_A.y[0], color=BOOK_COLORS['susceptible'], lw=1.5, label='S(t)')
ax.plot(sol_A.t, sol_A.y[1], color=BOOK_COLORS['infectious'], lw=1.5, label='I(t)')
ax.plot(sol_A.t, sol_A.y[2], color=BOOK_COLORS['recovered'], lw=1.5, label='R(t)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Fraction of population')
ax.set_title('SIR_I trajectory (baseline parameters, $R_0 = 2.0$)')
ax.legend()
ax.set_xlim(0, 100)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

print(f"Peak I: {sol_A.y[1].max():.4f} at t = {sol_A.t[sol_A.y[1].argmax()]:.1f} days")
print(f"Final R: {sol_A.y[2][-1]:.4f}  (fraction ever infected)")


## Verification: conservation and bounds

The SIR_I dynamics must satisfy:
1. $S + I + R = 1$ at every time step (population conservation)
2. $S, I, R \geq 0$ (non-negativity)
3. $S$ is monotone non-increasing
4. $R$ is monotone non-decreasing

In [ ]:
from shared.verification import assert_conservation_law

# Conservation check
assert_conservation_law([sol_A.y[0], sol_A.y[1], sol_A.y[2]], expected_total=1.0, tolerance=1e-8)
print("✅ Conservation: S + I + R = 1 to 1e-8")

# Non-negativity
assert sol_A.y.min() > -1e-10
print(f"✅ Non-negativity: min state value = {sol_A.y.min():.2e}")

# Monotonicity
assert (np.diff(sol_A.y[0]) <= 1e-12).all()
print("✅ S(t) monotone non-increasing")
assert (np.diff(sol_A.y[2]) >= -1e-12).all()
print("✅ R(t) monotone non-decreasing")


## What's next

The next notebook (`02_parameter_budget.ipynb`) introduces the canonical baseline parameter set used throughout the book. Then Chapter 6 develops the analytical properties: $R_0$, equilibria, and the invasion-burden partition.